<a href="https://colab.research.google.com/github/rhyan10/X-MACE/blob/X-MACE_socs/tutorial-summer-school.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# X-MACE Tutorial: Opening and Inspecting the Dataset

This notebook walks through:

1. Setting up the environment and fetching the data
2. Reading an extended XYZ file with ASE
3. Exploring basic `Atoms` properties
4. Understanding the `atoms.info` dictionary
5. Checking that every property array has a consistent shape
6. Iterating over all frames

The file is in **extended XYZ (extXYZ)** format — each frame's comment line carries the
reference properties, which ASE parses automatically into `atoms.info`.

Everything below is **derived from the file itself**: the number of atoms, the number of
electronic states, and which properties are present. Point `DATA_FILE` at a different
dataset and the whole notebook adapts.

## 0. Setup

Colab starts from a clean machine every time, so install ASE and fetch the data here.

In [ ]:
!pip install -q ase

import ase, ase.io
import numpy as np
print("ASE version:", ase.__version__)

### 0.1 Get the dataset

The cell tries, in order: a file already present, a download from `DATA_URL`, then a
manual upload. Set `DATA_URL` to a raw GitHub link so nobody has to upload anything.

In [ ]:
import os

DATA_FILE = "ch2nh2.xyz"
DATA_URL  = "https://raw.githubusercontent.com/rhyan10/X-MACE/X-MACE_socs/tutorials/ch2nh2.xyz"

if os.path.exists(DATA_FILE):
    print(f"Found {DATA_FILE} already.")
else:
    got = False
    if DATA_URL:
        rc = os.system(f"wget -q -O {DATA_FILE} {DATA_URL}")
        got = rc == 0 and os.path.getsize(DATA_FILE) > 0
        print("Downloaded." if got else "Download failed — falling back to upload.")
    if not got:
        if os.path.exists(DATA_FILE):
            os.remove(DATA_FILE)
        from google.colab import files
        files.upload()

assert os.path.exists(DATA_FILE), f"{DATA_FILE} is missing — cannot continue."
print("Size on disk:", round(os.path.getsize(DATA_FILE) / 1e6, 3), "MB")

## 1. Reading the XYZ File

Pass `":"` to read **all** frames, or `":10"` for the first 10 while exploring.

In [ ]:
db = ase.io.read(DATA_FILE, ":")
atoms = db[0]

print(f"Total frames loaded : {len(db)}")
print(f"Type of each element: {type(atoms)}")

## 2. Basic Atoms Properties

In [ ]:
print("Number of atoms    :", len(atoms))
print("Chemical formula   :", atoms.get_chemical_formula())
print("Chemical symbols   :", list(atoms.symbols))
print("Positions shape    :", atoms.positions.shape)   # (N_atoms, 3)

## 3. What's Actually in `atoms.info`?

`atoms.info` is a plain Python `dict`. Every key from the extXYZ comment line lands here,
already parsed into arrays. Different X-MACE datasets carry different properties — some
have spin-orbit couplings, some only energies, forces and NACs — so always look before
assuming.

In [ ]:
print(f"{'key':22s} {'shape':16s} dtype")
print("-" * 50)
for k, v in atoms.info.items():
    arr = np.asarray(v)
    print(f"{k:22s} {str(arr.shape):16s} {arr.dtype}")

## 4. Working Out the Dataset Dimensions

Rather than hardcoding numbers, read them off the data:

* `N` — atoms per frame
* `n_states` — electronic states, from the trailing axis of the energy array
* `n_pairs` — unique state pairs, `n_states x (n_states - 1) / 2`, which is how many
  NAC vectors there are

The cell also locates the NAC key, since datasets use `REF_nacs`, `REF_couplings`, or
`REF_smooth_nacs` depending on how they were generated.

In [ ]:
ENERGY_KEY = 'REF_energy'

energy   = np.array(atoms.info[ENERGY_KEY])
N        = len(atoms)
n_states = energy.shape[-1]
n_pairs  = n_states * (n_states - 1) // 2

# Find whichever NAC key this dataset uses. Naming is inconsistent across X-MACE
# datasets -- ch2nh2.xyz uses 'smooth_nacs' with no REF_ prefix, while
# SINGLET_SOC_ALL.xyz uses 'REF_smooth_nacs'. Detect by substring instead.
present_nacs = [k for k in atoms.info
                if ('nac' in k.lower() or 'coupling' in k.lower())]
present_nacs.sort(key=lambda k: ('smooth' not in k.lower(), k))   # prefer smoothed
NAC_KEY = present_nacs[0] if present_nacs else None

HAS_FORCES = 'REF_forces' in atoms.info
HAS_SOCS   = 'REF_socs'   in atoms.info

print(f"N_atoms   : {N}")
print(f"n_states  : {n_states}")
print(f"n_pairs   : {n_pairs}")
print(f"forces    : {'yes' if HAS_FORCES else 'no'}")
print(f"NAC key   : {NAC_KEY or 'none found'}"
      + (f"   (also present: {present_nacs[1:]})" if len(present_nacs) > 1 else ""))
print(f"SOCs      : {'yes' if HAS_SOCS else 'no'}")

### Expected shapes

| Key | Shape | Meaning |
|---|---|---|
| `REF_energy` | `(1, n_states)` | one geometry x state energies |
| `REF_forces` | `(N, n_states, 3)` | atoms x states x xyz |
| NAC key | `(N, n_pairs, 3)` | atoms x state pairs x xyz |
| `REF_socs` | `(1, n_soc)` | flat SOC vector, if present |

## 5. Inspecting Each Property

### 5.1 Energies

In [ ]:
print("shape  :", energy.shape)
print("values :", energy)

ev = np.sort(energy.ravel())
print("\nsorted :", ev)
print("S1-S0 gap at this geometry:", ev[1] - ev[0])

### 5.2 Forces

In [ ]:
if HAS_FORCES:
    forces = np.array(atoms.info['REF_forces'])
    print("forces shape :", forces.shape, f"  expected ({N}, {n_states}, 3)")

    # Re-order to (n_states, N_atoms, 3) for state-first indexing
    print("state-first  :", forces.transpose(1, 0, 2).shape)
    print("largest |force| in this frame:", np.abs(forces).max())
else:
    print("No forces in this dataset.")

### 5.3 Nonadiabatic Couplings

NACs are indexed by *state pair*, not by state. With `n_states` states there are
`n_pairs` of them, ordered (0,1), (0,2), ... The pair labels are generated below so you
can tell which coupling is which.

In [ ]:
from itertools import combinations

if NAC_KEY:
    nacs = np.array(atoms.info[NAC_KEY])
    print(f"{NAC_KEY} shape :", nacs.shape, f"  expected ({N}, {n_pairs}, 3)")

    if n_states == n_pairs:
        print(f"\nNote: with {n_states} states, n_pairs is also {n_pairs}, so forces and\n"
              "NACs have identical shapes. Don't rely on shape alone to tell them apart.")

    pair_labels = list(combinations(range(n_states), 2))
    print("\nstate pairs (column order):")
    for i, (a, b) in enumerate(pair_labels):
        print(f"  column {i}:  S{a} - S{b}   |max| = {np.abs(nacs[:, i, :]).max():.6f}")
else:
    print("No NAC key found in this dataset.")

### 5.4 Spin-Orbit Couplings

Only present in datasets that include triplets. Whatever length prints here is the value
to pass to `--soc_num` when training.

In [ ]:
if HAS_SOCS:
    socs = np.array(atoms.info['REF_socs'])
    print("socs shape :", socs.shape)
    print("n_soc      :", socs.size, " <- use for --soc_num")
else:
    print("No SOCs in this dataset — skip --compute_socs when training.")

## 6. Sanity Check Across All Frames

Checks every frame, not just the first, and only for the properties this dataset
actually has. Frames with a different atom count are reported rather than crashing.

In [ ]:
def check_frame(a, verbose=False):
    problems = []
    n = len(a)

    exp = {ENERGY_KEY: (1, n_states)}
    if HAS_FORCES:
        exp['REF_forces'] = (n, n_states, 3)
    if NAC_KEY:
        exp[NAC_KEY] = (n, n_pairs, 3)
    if HAS_SOCS:
        exp['REF_socs'] = np.array(db[0].info['REF_socs']).shape

    for key, want in exp.items():
        if key not in a.info:
            problems.append(f"{key}: missing")
            if verbose:
                print(f"  [--] {key:22s} missing")
            continue
        got = np.array(a.info[key]).shape
        if got != want:
            problems.append(f"{key}: expected {want}, got {got}")
        if verbose:
            print(f"  [{'OK ' if got == want else 'BAD'}] {key:22s} "
                  f"expected {str(want):16s} got {got}")
    return problems


print("Frame 0 in detail:")
check_frame(db[0], verbose=True)

print("\nScanning all frames...")
bad = {i: p for i, a in enumerate(db) if (p := check_frame(a))}

if not bad:
    print(f"All {len(db)} frames have consistent shapes.")
else:
    print(f"{len(bad)} of {len(db)} frames have problems. First few:")
    for i, probs in list(bad.items())[:5]:
        print(f"  frame {i}: {'; '.join(probs)}")

## 7. Iterating Over All Frames

One line per frame floods the output on a large dataset, so this previews a few and then
summarises everything.

In [ ]:
PREVIEW = 10

header = f"{'Frame':>6}  {'Min energy':>16}  {'S1-S0 gap':>14}"
if HAS_FORCES:
    header += f"  {'Max |force|':>14}"
print(header)
print("-" * len(header))

min_e, gaps, max_f = [], [], []
for i, frame in enumerate(db):
    e = np.sort(np.array(frame.info[ENERGY_KEY]).ravel())
    min_e.append(e[0])
    gaps.append(e[1] - e[0] if len(e) > 1 else np.nan)
    row = f"{i:>6}  {e[0]:>16.6f}  {gaps[-1]:>14.6f}"
    if HAS_FORCES:
        f_max = np.abs(np.array(frame.info['REF_forces'])).max()
        max_f.append(f_max)
        row += f"  {f_max:>14.6f}"
    if i < PREVIEW:
        print(row)

if len(db) > PREVIEW:
    print(f"... and {len(db) - PREVIEW} more frames")

min_e, gaps = np.array(min_e), np.array(gaps)
print(f"\nOver all {len(db)} frames:")
print(f"  ground-state energy : {min_e.min():.6f} to {min_e.max():.6f}")
print(f"  smallest S1-S0 gap  : {gaps.min():.6f}  (frame {gaps.argmin()})")
if HAS_FORCES:
    max_f = np.array(max_f)
    print(f"  largest |force|     : {max_f.max():.6f}  (frame {max_f.argmax()})")

## 8. Visualising the Conical Intersection Region

Near-zero S1-S0 gaps are the conical-intersection region — the non-smooth part of the
surface that X-MACE is built to represent. The right-hand plot shows why NACs matter:
they blow up exactly where the gap closes.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2 if NAC_KEY else 1, figsize=(11 if NAC_KEY else 6, 4),
                         squeeze=False)

axes[0][0].hist(gaps, bins=40)
axes[0][0].set_xlabel("S1 - S0 gap")
axes[0][0].set_ylabel("Number of frames")
axes[0][0].set_title("Energy gap distribution")

if NAC_KEY:
    nac_mag = np.array([np.abs(np.array(f.info[NAC_KEY])).max() for f in db])
    axes[0][1].scatter(gaps, nac_mag, s=14, alpha=0.7)
    axes[0][1].set_xlabel("S1 - S0 gap")
    axes[0][1].set_ylabel("max |NAC|")
    axes[0][1].set_yscale("log")
    axes[0][1].set_title("NAC magnitude vs gap")

plt.tight_layout()
plt.show()

print(f"Closest approach to a conical intersection: gap = {gaps.min():.6f} "
      f"at frame {gaps.argmin()}")